# Quality Control for Mol-JEPA Embeddings

This report checks the 512-dimensional classification-token embeddings from the [Mol-JEPA model](https://huggingface.co/Flogrammer/Mol-JEPA) and their coverage of the input Harmonized Drug Dataset (HDD) metadata. The embedding CSV and HDD labels are supplied by the workflow or `pixi run qc`.

The plots and classifiers are exploratory. Compound-level splits can put identical vectors from distinct HDD IDs in both train and test; use structure-aware splits for claims about generalization.

## Environment Set Up

We load in the environment, the embeddings we computed, and define a utility function for handling [ATC code levels](https://www.who.int/tools/atc-ddd-toolkit/atc-classification). 

In [ ]:
%matplotlib inline
import os
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import umap
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler


def process_atc_code(code_string, level):
    tokens = re.findall(r"[A-Za-z]|[^A-Za-z]+", code_string)
    return "".join(tokens[:level])


def describe_groups(frame, field, title):
    counts = frame[field].value_counts()
    display(counts.rename("Compounds"))
    if counts.empty:
        print(f"No matched labels for {title}.")
        return
    ax = sns.barplot(x=counts.index.astype(str), y=counts.values)
    ax.set(xlabel=field, ylabel="Compounds", title=title)
    ax.bar_label(ax.containers[0], padding=3)
    ax.margins(y=0.12)
    plt.show()


def preview_classifier(frame, label, title):
    counts = frame[label].value_counts()
    if len(counts) < 2 or counts.min() < 2:
        print(f"Not enough compounds per class to evaluate {title}.")
        return
    folds = min(20, int(counts.min()))
    X = frame[embedding_columns].to_numpy()
    y = LabelEncoder().fit_transform(frame[label])
    results = []
    split = StratifiedKFold(n_splits=folds)
    for fold, (train, test) in enumerate(split.split(X, y), start=1):
        models = {
            "RF": RandomForestClassifier(max_depth=2, random_state=0),
            "MLP": MLPClassifier(max_iter=1000, random_state=fold),
            "KNN": KNeighborsClassifier(n_neighbors=min(10, len(train))),
        }
        for name, model in models.items():
            model.fit(X[train], y[train])
            pred = model.predict(X[test])
            results.append({
                "Model": name,
                "Fold": fold,
                "Balanced accuracy": balanced_accuracy_score(y[test], pred),
                "Accuracy": accuracy_score(y[test], pred),
            })
    results = pd.DataFrame(results)
    display(results.groupby("Model")[["Accuracy", "Balanced accuracy"]].median().round(3))
    ax = sns.boxplot(data=results, x="Model", y="Balanced accuracy")
    ax.set(title=f"{title} ({len(frame)} compounds, {folds} folds)", ylim=(0, 1.03))
    plt.show()

In [ ]:
embeddings_file = Path(os.environ["MOLJEPA_QC_EMBEDDINGS"])
labels_file = Path(os.environ["MOLJEPA_QC_LABELS"])
data = pd.read_csv(embeddings_file, dtype={"HDD.Compound.ID": "string"})
id_col = "HDD.Compound.ID"
embedding_columns = [f"Emb-{i}" for i in range(1, 513)]
actual_columns = [c for c in data.columns if c.startswith("Emb-")]
if actual_columns != embedding_columns:
    raise ValueError("Expected ordered embedding columns Emb-1 through Emb-512.")
if data.empty or data[id_col].isna().any() or not data[id_col].is_unique:
    raise ValueError("Expected at least one row with unique, nonmissing HDD compound IDs.")
if not np.isfinite(data[embedding_columns].to_numpy()).all():
    raise ValueError("Embedding columns contain non-finite values.")
labels = pd.read_csv(
    labels_file,
    sep="\t",
    usecols=[id_col, "ATC.Code", "LTKB.DILI.Severity.Grade", "DICT.Cardiotoxicity"],
    dtype={id_col: "string"},
)
matched = data[[id_col]].merge(labels, on=id_col, how="left", validate="one_to_one", indicator=True)
print(f"{len(data):,} compounds with {len(embedding_columns)} features each.")
print(f"{sum(matched['_merge'] == 'both'):,} compounds found in source metadata.")
print(f"{data[embedding_columns].duplicated(keep=False).sum():,} compounds share an identical embedding with at least one other compound.")

# Visualizing the Embeddings


We first provide a visualization of how the Mol-JEPA embeddings cluster along level 1 ATC Code.





In [ ]:
atc_data = matched.dropna(subset=["ATC.Code"])[[id_col, "ATC.Code"]]
plot_df = pd.DataFrame()
if len(atc_data) >= 4:
    coords = data.merge(atc_data, on=id_col, validate="one_to_one")
    X = StandardScaler().fit_transform(coords[embedding_columns])
    reducer = umap.UMAP(random_state=42, n_neighbors=min(15, len(coords) - 1), init="random")
    embedding = reducer.fit_transform(X)
    plot_df = pd.DataFrame({"UMAP-1": embedding[:, 0], "UMAP-2": embedding[:, 1]})
    for level in (1, 2, 3):
        plot_df[f"ATC_L{level}"] = coords["ATC.Code"].map(lambda code: process_atc_code(code, level))
else:
    print(f"Only {len(atc_data)} compounds have an ATC label; skipping the projection.")

In [ ]:
def plot_atc(frame, field, title):
    if frame.empty:
        print(f"No compounds to plot for {title}.")
        return
    ax = sns.scatterplot(data=frame, x="UMAP-1", y="UMAP-2", hue=field)
    sns.move_legend(ax, "upper left", bbox_to_anchor=(1.02, 1))
    ax.set_title(title)
    plt.show()


plot_atc(plot_df, "ATC_L1", "Mol-JEPA embeddings by ATC level 1")

Explore the coverage of the ATC level 1 groups in the fitted projection.

In [ ]:
atc_data['ATC.Code'].map(lambda code: process_atc_code(code, 1)).value_counts()

Compare compounds in the nervous system (N) and antineoplastic and immunomodulating (L) groups.

In [ ]:
if not plot_df.empty:
    plot_atc(plot_df[plot_df["ATC_L1"].isin(["N", "L"])], "ATC_L1", "Nervous system and antineoplastic/immunomodulating compounds")

Zoom in on the L group, then on antineoplastic agents (L01). These views reuse the same projection and are not classification tests.

In [ ]:
if not plot_df.empty:
    display(plot_df.loc[plot_df["ATC_L1"] == "L", "ATC_L2"].value_counts())

Within L01, colour the projected compounds by ATC level 3.

In [ ]:
if not plot_df.empty:
    plot_atc(plot_df[plot_df["ATC_L2"] == "L01"], "ATC_L3", "Antineoplastic agents by ATC level 3")

Any apparent separation here is descriptive; the two-dimensional projection is not a performance estimate.

## Liver toxicity

Inspect the matched LTKB severity grades and compare exploratory classifiers for the binned label.

In [ ]:
grade = "LTKB.DILI.Severity.Grade"
liver = matched[[id_col, grade]].copy()
liver[grade] = pd.to_numeric(liver[grade], errors="coerce")
liver = liver.dropna(subset=[grade]).copy()
describe_groups(liver, grade, "Matched liver toxicity grades")

Treat grades 4 and higher as High; lower grades are Low.

In [ ]:
liver["Binned Tox"] = np.where(liver[grade] >= 4, "High", "Low")
describe_groups(liver, "Binned Tox", "Binned liver toxicity grades")

Show out-of-fold balanced accuracy and accuracy for the matched compounds.

In [ ]:
preview_classifier(data.merge(liver[[id_col, "Binned Tox"]], on=id_col, validate="one_to_one"), "Binned Tox", "Liver toxicity")

## Cardiotoxicity

Inspect DICT labels and run the same exploratory evaluation when class counts permit.

In [ ]:
cardio = "DICT.Cardiotoxicity"
cardiac = matched[[id_col, cardio]].dropna().copy()
describe_groups(cardiac, cardio, "Matched cardiotoxicity labels")

In [ ]:
preview_classifier(data.merge(cardiac, on=id_col, validate="one_to_one"), cardio, "Cardiotoxicity")

Performance in these checks applies to these compound-level splits only.